In [1]:
import datetime as dt
import time as tm

In [2]:
pisecond = dt.datetime(2021, 3, 14, 15, 9, 26)
print(pisecond)

2021-03-14 15:09:26


In [3]:
!pip install earthengine-api


  Using cached earthengine_api-1.5.24-py3-none-any.whl.metadata (2.1 kB)
  Using cached google_api_python_client-2.176.0-py3-none-any.whl.metadata (7.0 kB)
  Using cached google_auth_httplib2-0.2.0-py2.py3-none-any.whl.metadata (2.2 kB)
  Using cached httplib2-0.22.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached uritemplate-4.2.0-py3-none-any.whl.metadata (2.6 kB)
Using cached earthengine_api-1.5.24-py3-none-any.whl (464 kB)
Using cached httplib2-0.22.0-py3-none-any.whl (96 kB)
Using cached google_api_python_client-2.176.0-py3-none-any.whl (13.7 MB)
Using cached google_auth_httplib2-0.2.0-py2.py3-none-any.whl (9.3 kB)
Using cached uritemplate-4.2.0-py3-none-any.whl (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [earthengine-api] [earthengine-api]n-client]


In [ ]:
import ee
import folium
ee.Authenticate(force=True)
ee.Initialize()

In [ ]:
# Define AOI for Shanghai 
shanghai_bounds = ee.Geometry.Rectangle([120.8, 30.6, 122.2, 31.9])

#load the staellite data
image = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')\
.filterDate('2023-01-01', '2023-12-31')\
.filterBounds(shanghai_bounds)\
.sort('CLOUD_COVER')\
.first()

#make the training data
training = image.sample(**{
    'region': shanghai_bounds,
    'scale': 30,
    'numPixels': 5000
})

#Initiate the clusterer and train it
clusterer = ee.Clusterer.wekaKMeans(15).train(training)

#clusterer the input using the trained clusterer
result = image.cluster(clusterer)

#Display 
my_map.add_ee_layer(result.clip(shanghai_bounds).randomVisualizer(), {}, 'clusters')
display(my_map)

In [ ]:
#Using Random Forest approach for Shanghai Classification
# Define AOI for Shanghai 
shanghai_bounds = ee.Geometry.Rectangle([120.8, 30.6, 122.2, 31.9])

#load the staellite data
image = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')\
.filterDate('2023-01-01', '2023-12-31')\
.filterBounds(shanghai_bounds)\
.sort('CLOUD_COVER')\
.first()

#visualization
visParamsTrue = {'bands': ['B4', 'B3', 'B2'], 'min':0, 'max': 3000, 'gamma': 1.4}

#import training data
training = ee.FeatureCollection('users/midekisa/Train_Cover_CA')



In [1]:
import ee
import geemap
import geemap.colormaps as cm
import random

In [2]:
#Initialize Earth Engine
ee.Initialize()

In [12]:
# 3. Create an interactive map
Map = geemap.Map(center=[31.2, 121.5], zoom=9)

# 4. Define Shanghai region (bounding box or manually draw polygon)
shanghai = ee.Geometry.Rectangle([120.8, 30.6, 122.1, 31.9])

# 5. Load Landsat 8 Image Collection for 2017
landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
    .filterBounds(shanghai) \
    .filterDate('2017-01-01', '2017-12-31') \
    .filterMetadata('CLOUD_COVER', 'less_than', 20) \
    .median() \
    .clip(shanghai)

# 6. Select relevant bands for classification
bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7']
image = landsat.select(bands)

In [13]:
# 7. Define training polygons (manually or upload as feature collection)
# Example: Manually defining land cover samples (urban, vegetation, water)

urban = ee.Feature(ee.Geometry.Point([121.5, 31.2]), {'landcover': 0})
vegetation = ee.Feature(ee.Geometry.Point([121.3, 31.3]), {'landcover': 1})
water = ee.Feature(ee.Geometry.Point([121.6, 31.1]), {'landcover': 2})

training_samples = ee.FeatureCollection([urban, vegetation, water])

In [14]:
# 8. Sample image using training data
training = image.sampleRegions(
    collection=training_samples,
    properties=['landcover'],
    scale=30
)

# 9. Train a classifier (Random Forest)
classifier = ee.Classifier.smileRandomForest(numberOfTrees=50).train(
    features=training,
    classProperty='landcover',
    inputProperties=bands
)

# 10. Classify the image
classified = image.classify(classifier)